# 📊 Day 1: Tipe Data & Struktur Dataset

**Hari**: 1 dari 112 Hari Data Mining  
**Topik**: Tipe Data, Struktur Dataset, Pandas DataFrame Intro  
**Prerequisites**: Python dasar (variabel, list, dict, loop)  
**Estimasi Waktu**: 2–3 jam

---

## 🎯 Learning Objectives
Setelah menyelesaikan notebook ini, kamu akan:
- Membedakan tipe data: numerical, categorical, ordinal, nominal
- Memahami struktur dataset (baris = observasi, kolom = variabel)
- Membuat dan mengeksplor Pandas DataFrame
- Menggunakan `.info()`, `.describe()`, `.dtypes`, `.shape`
- Mengakses data dengan `.loc`, `.iloc`, dan column selection
- Mengubah tipe data dengan `.astype()`

In [ ]:
# ============================================================
# IMPORTS — Day 01: Tipe Data & Struktur Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)
pd.set_option('display.max_columns', None)

print('Setup selesai!')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')

---
## 🔍 1. Mengapa Tipe Data Penting?

Tipe data menentukan:
1. **Operasi matematika** apa yang boleh dilakukan
2. **Visualisasi** yang tepat
3. **Teknik preprocessing** yang harus dipakai
4. **Model ML** yang cocok

Contoh masalah jika salah tipe data:
- Menghitung rata-rata jenis kelamin → tidak masuk akal
- Menganggap 'Sangat Puas' = 3x 'Biasa' → asumsi jarak yang salah
- Memasukkan text langsung ke model numerik → error

---
## 🔍 2. Empat Tipe Data Utama

```
DATA
├── CATEGORICAL (Kualitatif)
│   ├── NOMINAL  → Kategori tanpa urutan (warna, kota, jenis kelamin)
│   └── ORDINAL  → Kategori dengan urutan (rating bintang, level pendidikan)
└── NUMERICAL (Kuantitatif)
    ├── DISCRETE   → Bilangan bulat terhitung (jumlah anak, jumlah kamar)
    └── CONTINUOUS → Nilai riil (tinggi badan, suhu, harga)
```

| Tipe | Contoh | Operasi Valid | Preprocessing Utama |
|------|--------|---------------|---------------------|
| **Nominal** | Warna, kota, merek | ==, != | One-Hot Encoding |
| **Ordinal** | Rating, level, ukuran baju | ==, <, > | Ordinal Encoding |
| **Discrete** | Jumlah item, hitungan | Semua aritmatika | Biasanya langsung |
| **Continuous** | Suhu, harga, berat | Semua aritmatika | Scaling / Normalisasi |

In [ ]:
# Demo: Contoh nyata setiap tipe data — Dataset Survei Kafe
data_survei = {
    # NOMINAL: tidak ada urutan
    'jenis_minuman': ['kopi', 'teh', 'kopi', 'jus', 'teh', 'kopi'],
    'metode_bayar':  ['cash', 'transfer', 'qris', 'cash', 'qris', 'transfer'],
    # ORDINAL: ada urutan, jarak tidak pasti
    'kepuasan':      ['sangat puas', 'puas', 'biasa', 'puas', 'sangat puas', 'tidak puas'],
    'ukuran_cup':    ['S', 'L', 'M', 'L', 'S', 'M'],
    # DISCRETE: bilangan bulat
    'jumlah_order':  [2, 1, 3, 1, 2, 4],
    # CONTINUOUS: nilai riil
    'total_bayar':   [45000.5, 22000.0, 87500.0, 18000.0, 41000.5, 105000.0],
    'waktu_tunggu':  [3.5, 2.1, 7.8, 1.5, 4.2, 9.1],
}
df_survei = pd.DataFrame(data_survei)
print('Dataset Survei Kafe:')
display(df_survei)
print()
print('Tipe data yang dikenali Pandas:')
print(df_survei.dtypes)

In [ ]:
# Visualisasi: Masing-masing tipe data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Visualisasi Berbagai Tipe Data — Survei Kafe', fontsize=16, fontweight='bold')

# NOMINAL — Bar chart
df_survei['jenis_minuman'].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['#2196F3','#4CAF50','#FF9800'], edgecolor='black')
axes[0,0].set_title('NOMINAL: Jenis Minuman', fontweight='bold')
axes[0,0].set_xlabel('Minuman'); axes[0,0].set_ylabel('Jumlah')
axes[0,0].tick_params(axis='x', rotation=0)

# ORDINAL — Bar chart dengan urutan yang benar!
urutan = ['tidak puas', 'biasa', 'puas', 'sangat puas']
df_survei['kepuasan'].value_counts().reindex(urutan, fill_value=0).plot(
    kind='bar', ax=axes[0,1], color=['#F44336','#FF9800','#4CAF50','#2196F3'], edgecolor='black')
axes[0,1].set_title('ORDINAL: Kepuasan (urutan penting!)', fontweight='bold')
axes[0,1].set_xlabel('Kepuasan'); axes[0,1].tick_params(axis='x', rotation=15)

# DISCRETE — Bar chart
df_survei['jumlah_order'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,0], color='#9C27B0', edgecolor='black')
axes[1,0].set_title('DISCRETE: Jumlah Order', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=0)

# CONTINUOUS — Histogram
axes[1,1].hist(df_survei['waktu_tunggu'], bins=5, color='#00BCD4', edgecolor='black', alpha=0.8)
axes[1,1].set_title('CONTINUOUS: Waktu Tunggu (menit)', fontweight='bold')
axes[1,1].set_xlabel('Menit'); axes[1,1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

---
## 🔍 3. Pandas DataFrame — Fondasi Data Mining

**DataFrame** = tabel 2D:
- **Baris** = satu observasi / record / sampel
- **Kolom** = satu variabel / fitur / atribut

### Cara Membuat DataFrame:

In [ ]:
# Metode 1: Dari dictionary
df1 = pd.DataFrame({'nama': ['Andi','Budi','Cici'], 'umur': [25,30,22], 'kota': ['Jakarta','Bandung','Surabaya']})
print('Dari Dictionary:'); display(df1)

# Metode 2: Dari list of dicts
df2 = pd.DataFrame([{'nama':'Dewi','nilai':85,'lulus':True},{'nama':'Eko','nilai':72,'lulus':True},{'nama':'Fani','nilai':58,'lulus':False}])
print('Dari List of Dicts:'); display(df2)

# Metode 3: Dataset bawaan seaborn
df = sns.load_dataset('tips')
print(f'Dataset Seaborn Tips: {df.shape[0]} baris x {df.shape[1]} kolom')
df.head(3)

---
## 🔍 4. Eksplorasi DataFrame — Fungsi Wajib Tahu

In [ ]:
df = sns.load_dataset('tips')

print('=== .shape ===')
print(f'{df.shape} → {df.shape[0]} baris x {df.shape[1]} kolom')
print()

print('=== .dtypes ===')
print(df.dtypes)
print()

print('=== .info() ===')
df.info()

In [ ]:
print('=== .describe() — Statistik Deskriptif Numerik ===')
display(df.describe().round(2))

print()
print('=== .describe(include="all") — Semua Kolom ===')
display(df.describe(include='all'))

In [ ]:
print('=== .value_counts() — Distribusi Kategorik ===')
print('Kolom day:')
print(df['day'].value_counts())
print()
print('Dalam persen:')
print((df['day'].value_counts(normalize=True)*100).round(1))
print()
print('=== .isnull().sum() — Missing Values ===')
print(df.isnull().sum())

---
## 🔍 5. Akses Data — loc, iloc, Boolean Filtering

In [ ]:
# Akses kolom
print('Satu kolom (Series):')
print(df['total_bill'].head())

print()
print('Beberapa kolom (DataFrame):')
display(df[['total_bill', 'tip', 'size']].head(3))

In [ ]:
# .loc — akses by LABEL
print('.loc[0:3, ["total_bill","tip","sex"]]:')
display(df.loc[0:3, ['total_bill','tip','sex']])

# .iloc — akses by POSISI
print()
print('.iloc[0:3, 0:3]:')
display(df.iloc[0:3, 0:3])

In [ ]:
# Boolean Filtering
print('Pelanggan dengan tip > 5:')
df_tip_besar = df[df['tip'] > 5]
print(f'Jumlah: {len(df_tip_besar)} dari {len(df)} pelanggan')
display(df_tip_besar.head(3))

print()
print('Makan malam di hari Jumat:')
mask = (df['time'] == 'Dinner') & (df['day'] == 'Fri')
display(df[mask])

---
## 🔍 6. Mengubah Tipe Data dengan .astype()

In [ ]:
# Contoh konversi tipe data
df_kotor = pd.DataFrame({
    'umur':       ['25', '30', '22', '45'],
    'pendapatan': ['5000000', '8000000', '3500000', '12000000'],
    'aktif':      ['True', 'False', 'True', 'True'],
    'level':      ['junior', 'senior', 'junior', 'lead'],
})

print('Sebelum konversi:'); print(df_kotor.dtypes); print()

df_bersih = df_kotor.copy()
df_bersih['umur']       = df_bersih['umur'].astype(int)
df_bersih['pendapatan'] = df_bersih['pendapatan'].astype(int)
df_bersih['aktif']      = df_bersih['aktif'].map({'True': True, 'False': False})

level_order = pd.CategoricalDtype(categories=['junior','senior','lead'], ordered=True)
df_bersih['level'] = df_bersih['level'].astype(level_order)

print('Sesudah konversi:'); print(df_bersih.dtypes)
print()
print('Test ordinal: senior > junior?', df_bersih['level'][1] > df_bersih['level'][0])
display(df_bersih)

---
## ⚠️ 7. Common Mistakes

1. **Angka != Numerik** — Kode pos, ID pelanggan, nomor KTP adalah angka tapi NOMINAL (tidak bisa dijumlah dengan makna)
2. **Ordinal != Interval** — Jarak antara bintang 1 dan 2 belum tentu sama dengan 4 dan 5
3. **`df.dtypes` bukan tipe statistik** — Pandas hanya tahu format penyimpanan, bukan makna datanya
4. **Selalu gunakan domain knowledge** — Konteks bisnis/domain paling penting untuk menentukan tipe data

```python
# Pertanyaan panduan menentukan tipe data:
# Q: Apakah bisa dijumlah dengan makna?     → Ya = Numerik
# Q: Apakah ada urutan yang bermakna?       → Ya = Ordinal, Tidak = Nominal
# Q: Apakah nilainya selalu bilangan bulat? → Ya = Discrete, Tidak = Continuous
```

In [ ]:
# Analisis Tipe Data Dataset Titanic — Praktik Domain Knowledge
df_titanic = sns.load_dataset('titanic')

print('Dataset Titanic — Analisis Tipe Data')
print('=' * 55)
print(f'Total kolom numerik  : {df_titanic.select_dtypes(include="number").shape[1]}')
print(f'Total kolom non-num  : {df_titanic.select_dtypes(exclude="number").shape[1]}')
print()

klasifikasi = {
    'survived': 'NOMINAL/BINARY — 0=meninggal, 1=selamat (tidak bisa dirata-rata!)',
    'pclass':   'ORDINAL     — 1=mewah > 2=menengah > 3=ekonomi',
    'sex':      'NOMINAL     — male/female, tidak ada urutan',
    'age':      'CONTINUOUS  — usia dalam tahun, nilai riil',
    'sibsp':    'DISCRETE    — jumlah saudara/pasangan di kapal',
    'fare':     'CONTINUOUS  — harga tiket, nilai riil',
    'embarked': 'NOMINAL     — pelabuhan keberangkatan (C/Q/S)',
}
print('Klasifikasi berdasarkan domain knowledge:')
print('-' * 55)
for kol, ket in klasifikasi.items():
    print(f'  {kol:10s}: {ket}')

---
## 📌 8. Ringkasan

### Cheatsheet Fungsi Pandas Hari Ini:
```python
df.shape                      # (n_baris, n_kolom)
df.head(n) / df.tail(n)       # n baris pertama/terakhir
df.info()                     # ringkasan struktur
df.dtypes                     # tipe data setiap kolom
df.describe()                 # statistik deskriptif numerik
df.describe(include='all')    # termasuk kategorik
df['kolom'].value_counts()    # distribusi kategorik
df.isnull().sum()             # missing value per kolom
df.loc[baris, kolom]          # akses by label
df.iloc[baris, kolom]         # akses by posisi integer
df[df['kol'] > nilai]         # boolean filtering
df['kol'].astype(tipe)        # konversi tipe data
```

---
## 🏋️ Selanjutnya: Latihan!

Buka folder `latihan/` dan kerjakan:
- **exercise_1.ipynb** — Identifikasi tipe data
- **exercise_2.ipynb** — Buat DataFrame dari berbagai sumber
- **exercise_3.ipynb** — Statistik deskriptif & eksplorasi
- **exercise_4.ipynb** — Deteksi inkonsistensi data
- **exercise_5.ipynb** — Mini EDA dataset Titanic

Solusi tersedia di folder `solusi/` jika sudah mencoba sendiri.